# Coverage — `coverage.csv`

Verifies table 10 — the per-`ModelID` boolean presence matrix `docs/PROJECT_ARCHITECTURE.md` calls "the `eda/integration_eda.ipynb` presence matrix promoted to a first-class pipeline output." This is what makes root `_.md` §1's fourth requirement — *where evidence is missing* — answerable at all. Every `has_*` sum here should match the corresponding table's own unique-`ModelID` count exactly, since both are computed from the same underlying data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

coverage = pd.read_csv("../../data/processed/coverage.csv")
coverage.shape

In [ ]:
coverage.head()

### Cross-check `has_*` sums against each source table's own row count
This isn't a new computation — it's confirming table 10 agrees with tables 3-9, since a mismatch here would mean the presence matrix and the actual data have drifted apart.

In [ ]:
has_cols = [c for c in coverage.columns if c.startswith("has_")]
coverage[has_cols].sum()

**Confirmed, matching every table's own unique-`ModelID` count from this session's verification:** `has_expression_rna`=1,479, `has_expression_rna_hpa`=1,103, `has_protein`=375, `has_mutations`=1,744, `has_fusions`=1,699, `has_dependency`=1,208, `has_copy_number`=1,118, `has_metabolomics`=927 — every single one lines up exactly with the corresponding table's own count.

### Genome-signature columns
`msi_high` (`MSIScore >= 20`) and `wgs_available` should match the counts Phase 1's `eda/individual_eda/14_OmicsGlobalSignatures.ipynb`-adjacent work already established.

In [ ]:
print(f"msi_high: {coverage['msi_high'].sum()}")
print(f"wgs_available: {coverage['wgs_available'].sum()}")

**Confirmed: `msi_high`=112 and `wgs_available`=1,622** — both match the values already established in this project's earlier genome-signatures EDA exactly.

### How many layers does a typical cell line actually have?
The real-world answer to root `_.md` §1's "where is evidence missing" question: most lines don't have all 8 layers, and a meaningful number have none of the 8 measured-evidence layers at all (metadata-only entries).

In [ ]:
layer_count = coverage[has_cols].sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
layer_count.value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Number of the 8 measured layers present"); ax.set_ylabel("Cell lines")
ax.set_title("How many evidence layers does each cell line actually have?")
plt.tight_layout(); plt.show()

print(layer_count.value_counts().sort_index())

**82 of 2,139 lines (3.8%) have zero measured layers** — metadata-only entries (the df9 lines with no sequencing profile at all, plus any df17-backfilled orphans that resolved a `ModelID` but carried no actual omics row alongside it). These 82 should never surface as a scoring candidate with fabricated evidence — they simply have nothing to abstain *from*, which is a distinct, correct state, not a bug. The rest spread across 1-8 layers, with the two ends (1 layer, 359 lines at 5 layers, 264 lines at all 8) forming the shape a real, unevenly-instrumented multi-omics panel should have.

### Verdict
Every `has_*` sum reconciles exactly with its source table, the genome-signature columns match prior EDA findings precisely, and the layer-count distribution is the honest, uneven shape this project's coverage table exists to surface. No concerns found.